# 00 — Setup Unity Catalog and paths

Run this **once** (or after changing catalog / storage names) before bronze ingest.

Creates:
- Catalog `sample_pipeline`
- Schemas `bronze`, `silver`, `gold`
- Documents ADLS landing paths and optional secret names for Drive / Confluence

**Edit the Config cell** to match your storage account, container, and secret scope.

## 1. Config

In [ ]:
# --- edit these for your workspace ---
CATALOG = "sample_pipeline"

STORAGE_ACCOUNT = "stpracticeecom"          # same pattern as Autoloader lab
CONTAINER = "practice-ecommerce"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Where seed / landed files live under ADLS (upload CSVs here before bronze)
LANDING_ADLS_ORDERS = f"{BASE_PATH}/landing/sample_pipeline/adls/orders/"
LANDING_ADLS_ORDER_ITEMS = f"{BASE_PATH}/landing/sample_pipeline/adls/order_items/"
LANDING_DRIVE_PRODUCTS = f"{BASE_PATH}/landing/sample_pipeline/google_drive/products/"
LANDING_CONFLUENCE_CUSTOMERS = f"{BASE_PATH}/landing/sample_pipeline/confluence/customers/"

# Optional: stage copies pulled live from Drive / Confluence APIs
STAGE_DRIVE = f"{BASE_PATH}/landing/sample_pipeline/_stage/google_drive/"
STAGE_CONFLUENCE = f"{BASE_PATH}/landing/sample_pipeline/_stage/confluence/"

# Secret scope (create in Databricks; leave unused until bronze API cells)
SECRET_SCOPE = "sample_pipeline"
# Expected secret keys (create later if using live APIs):
#   google_drive_api_key or service-account JSON path pattern
#   confluence_base_url, confluence_email, confluence_api_token

print("CATALOG:", CATALOG)
print("BASE_PATH:", BASE_PATH)
print("SECRET_SCOPE:", SECRET_SCOPE)

## 2. Create catalog and schemas

Requires permission to `CREATE CATALOG` (or ask an admin to create `sample_pipeline` first, then run the schema cells).

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'Sample multi-source medallion pipeline (ADLS + Drive + Confluence)'")
spark.sql(f"USE CATALOG {CATALOG}")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Raw land from ADLS, Google Drive, Confluence'")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Cleansed, deduped, FK-validated'")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Business aggregates for dashboards'")

print("Catalog and schemas ready.")

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## 3. Ensure landing directories exist

Creates empty folders under ADLS so uploads have a target. Needs storage access (same credential as Autoloader).

In [ ]:
landing_dirs = [
    LANDING_ADLS_ORDERS,
    LANDING_ADLS_ORDER_ITEMS,
    LANDING_DRIVE_PRODUCTS,
    LANDING_CONFLUENCE_CUSTOMERS,
    STAGE_DRIVE,
    STAGE_CONFLUENCE,
]

for path in landing_dirs:
    dbutils.fs.mkdirs(path)
    print("ok:", path)

## 4. Upload checklist (repo → ADLS)

From the repo folder `data/`, upload seed files into the landing paths:

| Local file | ADLS destination folder |
|---|---|
| `data/adls/orders.csv` | `.../landing/sample_pipeline/adls/orders/` |
| `data/adls/order_items.csv` | `.../landing/sample_pipeline/adls/order_items/` |
| `data/google_drive/products.csv` | `.../landing/sample_pipeline/google_drive/products/` |
| `data/confluence/customers.csv` | `.../landing/sample_pipeline/confluence/customers/` |

For the lab you can upload via Azure Portal, Storage Explorer, or `dbutils.fs.cp` from a Volume / DBFS upload.

Later, bronze can also **pull** Drive / Confluence live and write into `_stage/` — landing copies above still work as a fallback if APIs are blocked.

## 5. Optional — copy seeds from a UC Volume / DBFS upload path

If you uploaded the four CSVs to a workspace path first, set `SEED_ROOT` and run. Otherwise skip.

In [ ]:
# Example: after uploading the data/ folder to a Volume
# SEED_ROOT = "/Volumes/sample_pipeline/bronze/seeds"
SEED_ROOT = None  # set to a path string to enable copy

if SEED_ROOT:
    copies = [
        (f"{SEED_ROOT}/adls/orders.csv", f"{LANDING_ADLS_ORDERS}orders.csv"),
        (f"{SEED_ROOT}/adls/order_items.csv", f"{LANDING_ADLS_ORDER_ITEMS}order_items.csv"),
        (f"{SEED_ROOT}/google_drive/products.csv", f"{LANDING_DRIVE_PRODUCTS}products.csv"),
        (f"{SEED_ROOT}/confluence/customers.csv", f"{LANDING_CONFLUENCE_CUSTOMERS}customers.csv"),
    ]
    for src, dst in copies:
        dbutils.fs.cp(src, dst, True)
        print(f"copied {src} -> {dst}")
else:
    print("SEED_ROOT not set — upload CSVs to ADLS landing paths manually (see checklist above).")

## 6. Verify ADLS landing access

In [ ]:
def list_or_warn(path: str):
    try:
        files = dbutils.fs.ls(path)
        print(f"\n{path}")
        if not files:
            print("  (empty — upload seed CSV here before bronze)")
        for f in files:
            print(f"  {f.name}\t{f.size}")
    except Exception as e:
        print(f"\nFAILED {path}\n  {e}")

for p in [
    LANDING_ADLS_ORDERS,
    LANDING_ADLS_ORDER_ITEMS,
    LANDING_DRIVE_PRODUCTS,
    LANDING_CONFLUENCE_CUSTOMERS,
]:
    list_or_warn(p)

## 7. Optional — secret scope probe

Does not fail the notebook if secrets are missing. Bronze can run from ADLS-landed seeds without them.

In [ ]:
expected_secrets = [
    "confluence_base_url",
    "confluence_email",
    "confluence_api_token",
    "google_drive_file_id",  # products CSV / Sheet export id
]

print(f"Scope: {SECRET_SCOPE}")
for key in expected_secrets:
    try:
        _ = dbutils.secrets.get(SECRET_SCOPE, key)
        print(f"  OK  {key}")
    except Exception:
        print(f"  --  {key} (not set — fine for seed-file bronze)")

## 8. Persist config summary

Print values to copy into `01_bronze_ingest` (same constants).

In [ ]:
summary = {
    "catalog": CATALOG,
    "bronze": f"{CATALOG}.bronze",
    "silver": f"{CATALOG}.silver",
    "gold": f"{CATALOG}.gold",
    "landing": {
        "orders": LANDING_ADLS_ORDERS,
        "order_items": LANDING_ADLS_ORDER_ITEMS,
        "products": LANDING_DRIVE_PRODUCTS,
        "customers": LANDING_CONFLUENCE_CUSTOMERS,
    },
    "secret_scope": SECRET_SCOPE,
}

display(spark.createDataFrame([
    ("catalog", summary["catalog"]),
    ("bronze", summary["bronze"]),
    ("silver", summary["silver"]),
    ("gold", summary["gold"]),
    ("orders_path", summary["landing"]["orders"]),
    ("order_items_path", summary["landing"]["order_items"]),
    ("products_path", summary["landing"]["products"]),
    ("customers_path", summary["landing"]["customers"]),
    ("secret_scope", summary["secret_scope"]),
], ["key", "value"]))

print("Setup complete. Next: upload seeds (if needed), then run 01_bronze_ingest.ipynb")